# Twitter Research Agent with GetXAPI and Claude

This notebook builds a Claude agent that researches a topic on Twitter / X by giving the model two tools backed by the [GetXAPI](https://www.getxapi.com) REST API:

1. `search_tweets` — query Twitter / X for recent tweets matching a search string
2. `get_user_info` — look up a user profile by username

Claude decides which tool to call, when to stop, and how to synthesize the results. The cookbook demonstrates:

- Wrapping a third-party REST API as Claude tools
- The tool-use loop (`messages.create` with `tools=`, then re-invoking with `tool_result` content)
- Auditing what the agent did by inspecting tool inputs and outputs

**Authentication.** GetXAPI uses a single bearer token (`Authorization: Bearer <GETXAPI_KEY>`). Read endpoints — search, user info, followers, replies, mentions, timeline — only need this key. Write endpoints (post, reply, like, retweet, DM, articles) need an additional X account session token, which we do not use in this notebook.

**Prerequisites.** A GetXAPI key from [getxapi.com](https://www.getxapi.com) and an Anthropic API key from [console.anthropic.com/settings/keys](https://console.anthropic.com/settings/keys), both set in a local `.env` file (see `.env.example`).

## Setup

Install dependencies and load API keys from the environment.

In [ ]:
%pip install --quiet --upgrade anthropic requests python-dotenv

In [ ]:
import json
import os

import requests
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

GETXAPI_KEY = os.environ["GETXAPI_KEY"]
anthropic_client = Anthropic()  # picks up ANTHROPIC_API_KEY from env

GETXAPI_BASE = "https://api.getxapi.com"
GETXAPI_HEADERS = {"Authorization": f"Bearer {GETXAPI_KEY}"}
MODEL = "claude-sonnet-4-5"

## Define the GetXAPI helpers

Two thin functions that call the GetXAPI REST endpoints. Both use the same bearer auth. Errors are returned as a JSON-serializable dict so the model can read them.

Endpoints used:

- `GET /twitter/tweet/advanced_search` — params: `q` (required), `product` (`Top` | `Latest` | `People`, optional), `cursor` (optional)
- `GET /twitter/user/info` — params: `userName` (required)

In [ ]:
def search_tweets(query: str, product: str = "Latest") -> dict:
    """Search tweets matching `query`. Returns the API response body."""
    resp = requests.get(
        f"{GETXAPI_BASE}/twitter/tweet/advanced_search",
        headers=GETXAPI_HEADERS,
        params={"q": query, "product": product},
        timeout=30,
    )
    if not resp.ok:
        return {"error": f"HTTP {resp.status_code}", "body": resp.text[:500]}
    return resp.json()


def get_user_info(user_name: str) -> dict:
    """Look up a user profile by username (no leading @)."""
    resp = requests.get(
        f"{GETXAPI_BASE}/twitter/user/info",
        headers=GETXAPI_HEADERS,
        params={"userName": user_name.lstrip("@")},
        timeout=30,
    )
    if not resp.ok:
        return {"error": f"HTTP {resp.status_code}", "body": resp.text[:500]}
    return resp.json()

## Define the tools Claude can call

Each tool has a `name`, a `description` the model uses to decide when to call it, and an `input_schema` describing the parameters.

In [ ]:
TOOLS = [
    {
        "name": "search_tweets",
        "description": (
            "Search recent tweets on Twitter / X. Use this to find what people are "
            "saying about a topic, product, event, or person. Returns a list of tweets "
            "with author, text, and engagement counts."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Search query. Twitter / X search operators are supported "
                        "(e.g. `from:elonmusk`, `since:2026-05-01`, `-filter:replies`)."
                    ),
                },
                "product": {
                    "type": "string",
                    "enum": ["Top", "Latest", "People"],
                    "description": (
                        "Result ordering. `Top` returns Twitter's relevance picks, "
                        "`Latest` returns chronological, `People` returns user matches."
                    ),
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "get_user_info",
        "description": (
            "Look up a Twitter / X user profile by username. Use this after finding "
            "an interesting author in search results to understand who they are."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "user_name": {
                    "type": "string",
                    "description": "Username, with or without leading `@`.",
                },
            },
            "required": ["user_name"],
        },
    },
]


def dispatch_tool(name: str, tool_input: dict) -> str:
    """Call the matching helper and return the result as a JSON string."""
    if name == "search_tweets":
        result = search_tweets(
            query=tool_input["query"],
            product=tool_input.get("product", "Latest"),
        )
    elif name == "get_user_info":
        result = get_user_info(user_name=tool_input["user_name"])
    else:
        result = {"error": f"unknown tool: {name}"}
    return json.dumps(result)[:8000]  # trim very large responses

## The tool-use loop

The loop is the same shape as every Claude tool-use example:

1. Send the conversation + tool definitions to `messages.create`.
2. If `stop_reason == "end_turn"`, return the final text.
3. If `stop_reason == "tool_use"`, execute every `tool_use` block in the response, append the assistant message and a user message containing the matching `tool_result` blocks, then loop.

We print each tool call as it happens so you can audit the agent's reasoning trail.

In [ ]:
def run_agent(task: str, max_turns: int = 8) -> str:
    messages = [{"role": "user", "content": task}]

    for turn in range(max_turns):
        response = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=2048,
            tools=TOOLS,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return "".join(
                block.text for block in response.content if block.type == "text"
            )

        if response.stop_reason != "tool_use":
            return f"[stopped: {response.stop_reason}]"

        # Echo any interim text Claude produced before the tool call.
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print(f"[turn {turn}] {block.text.strip()}")

        # Run every tool_use block and collect tool_result blocks.
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"[turn {turn}] -> {block.name}({json.dumps(block.input)})")
                result_str = dispatch_tool(block.name, block.input)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_str,
                    }
                )

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    return "[max_turns reached]"

## Run the agent

Give it an open-ended research task. The agent decides how many searches to run, whether to look up specific users, and when it has enough to answer.

In [ ]:
task = (
    "Find recent tweets about Claude agents and summarize the three most "
    "interesting things people are building. For each, name the author and "
    "include a one-sentence description."
)

summary = run_agent(task)
print("\n--- Final answer ---\n")
print(summary)

## Where to go next

- **More tools.** GetXAPI exposes 47 endpoints — tweet replies, user followers, verified followers, user timelines, mentions, bookmarks, lists, articles, DMs. Add another helper and another tool definition and Claude can use it the same way.
- **Write actions.** Post tweets, reply, like, retweet, send DMs, publish articles. These need an X account session token in addition to the GetXAPI key — see the [docs](https://docs.getxapi.com) for the write-tool auth flow.
- **MCP variant.** The same API is also packaged as an MCP server at [github.com/getxapi/getxapi-mcp](https://github.com/getxapi/getxapi-mcp), which lets Claude Desktop, Cursor, and other MCP clients use these endpoints without you wiring tool definitions by hand.
- **Scheduling.** Wrap `run_agent()` in a cron job to produce a daily digest, a launch-tracker alert, or a competitor-mention summary.